# Etapa 6: Modelado e Inferencia Estadística Dual (Paramétrica vs No Paramétrica)

**Proyecto**: Plataforma Analítica del Mercado de la Papa en Colombia (SIPSA 2019–2025)  
**Marco Metodológico**: CRISP-DM (Etapa 6: Modeling / Evaluation) & Marco PDCO (Fase DEVELOPMENT)  
**Estándares**: Clean Architecture, SWEBOK, DAMA-BOK, ISO/IEC 25010  

---

### Objetivos de la Etapa:
1. **Canal Paramétrico**: Contrastes de Levene (homocedasticidad), ANOVA clásico de un factor, ANOVA de Welch (robusto ante heterocedasticidad) e intervalos t-Student al 95%.
2. **Canal No Paramétrico**: Prueba de Fligner-Killeen, prueba de rangos de Kruskal-Wallis e Intervalos de Confianza Bootstrap BCa (Bias-Corrected and Accelerated) al 95% ($B=2,000$).
3. **Contraste Dual**: Evaluación empírica de sesgo entre media paramétrica y mediana robusta.
4. **Persistencia Medallion**: Exportación a `data/CURATED/contrastes_estadisticos_anuales.parquet`.

In [ ]:
# ==============================================================================
# CELDA 1 OBLIGATORIA: INSTALACIÓN IDEMPOTENTE DE DEPENDENCIAS EN EL KERNEL
# ==============================================================================
import sys

%pip install --quiet --upgrade pip
%pip install --quiet \
    numpy>=1.24.0 \
    pandas>=2.0.0 \
    scipy>=1.10.0 \
    fastparquet>=2023.8.0 \
    tabulate>=0.9.0

print(f"[OK] Ambiente verificado en Python: {sys.version.split()[0]}")

In [ ]:
# ==============================================================================
# CONFIGURACIÓN DE IMPORTS Y RUTAS DEL PROYECTO (CLEAN ARCHITECTURE)
# ==============================================================================
from pathlib import Path
import sys

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

from src.infrastructure.config import FEATURES_DIR, CURATED_DIR
from src.application.model_service import ModelService

print("Servicio de Modelado Estadístico Dual cargado exitosamente.")

In [ ]:
# ==============================================================================
# CARGA DE DATOS DE LA CAPA FEATURES
# ==============================================================================
parquet_path = FEATURES_DIR / "dataset_sipsa_features.parquet"

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
else:
    # Fallback con múltiples años para contrastes ANOVA y Kruskal-Wallis
    rows = []
    for año in [2021, 2022, 2023, 2024]:
        media_año = 1800 if año == 2021 else (2600 if año == 2022 else (2300 if año == 2023 else 2800))
        precios = np.random.gamma(shape=8.0, scale=media_año / 8.0, size=40)
        for p in precios:
            rows.append({
                "año": año,
                "precio_prom_kg": float(p),
                "volumen_ingreso_ton": float(np.random.uniform(50, 200)),
                "variedad_papa": "PAPA PASTUSA"
            })
    df = pd.DataFrame(rows)

print(f"Registros disponibles para contrastes de hipótesis: {len(df):,}")
display(df.head(4))

In [ ]:
# ==============================================================================
# EJECUCIÓN DE LA BATERÍA ESTADÍSTICA DUAL
# ==============================================================================
mod_service = ModelService(df, output_dir=CURATED_DIR)
df_tabla_dual, resumen = mod_service.compilar_y_guardar_resultados(col_grupo="año", col_valor="precio_prom_kg")

print("=== CONTRASTES PARAMÉTRICOS (ANOVA & LEVENE) ===")
print(f"Levene W: {resumen['canal_parametrico']['prueba_levene']['estadistico_W']:.3f} | p-valor: {resumen['canal_parametrico']['prueba_levene']['p_valor']:.4e}")
print(f"ANOVA F: {resumen['canal_parametrico']['anova_clasico']['estadistico_F']:.3f} | p-valor: {resumen['canal_parametrico']['anova_clasico']['p_valor']:.4e}")
print(f"Welch F: {resumen['canal_parametrico']['anova_welch']['estadistico_F_welch']:.3f} | p-valor: {resumen['canal_parametrico']['anova_welch']['p_valor']:.4e}")

print("\n=== CONTRASTES NO PARAMÉTRICOS (KRUSKAL-WALLIS & FLIGNER) ===")
print(f"Fligner Chi2: {resumen['canal_no_parametrico']['prueba_fligner_killeen']['estadistico_chi2']:.3f} | p-valor: {resumen['canal_no_parametrico']['prueba_fligner_killeen']['p_valor']:.4e}")
print(f"Kruskal H: {resumen['canal_no_parametrico']['kruskal_wallis']['estadistico_H']:.3f} | p-valor: {resumen['canal_no_parametrico']['kruskal_wallis']['p_valor']:.4e}")

print("\n=== TABLA COMPARATIVA: ESTIMADORES DUALES E INTERVALOS AL 95% ===")
display(df_tabla_dual)

### Conclusiones de la Etapa 6:
- Si la prueba de Levene rechaza homocedasticidad ($p < 0.05$), el **ANOVA de Welch** y la prueba de **Kruskal-Wallis** constituyen los contrastes con validez estadística.
- Los intervalos Bootstrap BCa (95%) capturan con fidelidad la asimetría de la distribución sin requerir supuestos gaussianos.
- Los resultados consolidados se encuentran persistidos en `data/CURATED/contrastes_estadisticos_anuales.parquet` para alimentar la **Etapa 7 (Visualization)**.